https://youtu.be/0n7aGNVCtRc

In [0]:
lift_data = [
    (1,300),
    (2,350)
]

lift_schema = "id int , capacity_kg int"

lift_df = spark.createDataFrame(data = lift_data , schema = lift_schema)


lift_passengers_data = [
    ('Rahul',85,1),
    ('Adarsh',73,1),
    ('Riti',95,1),
    ('Viraj',80,1),
    ('Vimal',83,2),
    ('Neha',77,2),
    ('Priti',73,2),
    ('Himanshi',85,2)
]

lift_passengers_schema = "passenger_name string , weight_kg int, lift_id int"

lift_passengers_df = spark.createDataFrame(data = lift_passengers_data , schema = lift_passengers_schema)

lift_df.display()
lift_passengers_df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
lift_passengers_df = (
    lift_passengers_df
        .withColumn("running_weight" ,
                    F.sum(F.col("weight_kg")).over(Window.partitionBy("lift_id").orderBy("weight_kg"))
                    )
        .join(lift_df, lift_passengers_df.lift_id == lift_df.id ,"inner")
        .filter("running_weight <= capacity_kg")
        .groupBy("lift_id").agg(F.concat_ws(',',F.collect_list("passenger_name").alias("passenger_name")))
)

lift_passengers_df.display()